# FIREGUARD ID — 03: Model Design (Advanced ML: HDBSCAN)
**Tujuan Minggu 3 (Bagian 2):** Melatih Model 2 (Advanced ML) dan menjustifikasinya berdasarkan teori, keunggulan/kelemahan, karakteristik data, *hypothesis space*, dan *inductive bias*, lalu membandingkannya dengan Baseline pada **Validation Set**.

---

## Bab 6.2.1 Teori, Keunggulan, & Kelemahan Model Advanced (HDBSCAN)

### Teori Singkat
**HDBSCAN** (*Hierarchical Density-Based Spatial Clustering of Applications with Noise*) memperluas DBSCAN dengan mengubah ruang jarak menjadi *mutual reachability distance*, membangun *Minimum Spanning Tree (MST)*, membentuk hierarki klaster, dan mengekstrak klaster paling stabil menggunakan metode *Excess of Mass* (`eom`).

### Alasan Pemilihan Berdasarkan Karakteristik Data
Data grid Karhutla Riau memiliki distribusi **sangat miring ke kanan (*right-skewed*)** dan **ketidakseimbangan spasial (*spatial imbalance*)**: sebagian besar grid (560+ unit) bernilai mendekati nol, sementara segelintir grid episentrum memiliki kepadatan hotspot dan intensitas FRP sangat tinggi dengan jarak bervariasi. HDBSCAN dipilih karena tidak memaksa seluruh titik ekstrem masuk ke bentuk klaster sferis.

### Keunggulan & Kelemahan
- **Keunggulan:**
  1. Tidak memerlukan penetapan jumlah klaster $k$ secara manual di awal.
  2. Mampu mendeteksi klaster dengan bentuk non-sferis dan kepadatan yang berbeda-beda.
  3. Dapat mengisolasi anomali/titik berdensitas rendah sebagai *noise* (`label = -1`).
- **Kelemahan:**
  1. Sensitif terhadap parameter `min_cluster_size` dan `min_samples`; pada data dengan ketimpangan ekstrem, proporsi *noise* dapat menjadi tinggi.
  2. Implementasi `sklearn.cluster.HDBSCAN` tidak menyediakan fungsi `predict()` bawaan untuk data baru sehingga evaluasi Validation Set memerlukan pendekatan *nearest-centroid assignment* dari klaster inti.

## Bab 6.2.2 Hypothesis Space & Inductive Bias
- **Hypothesis Space:** Himpunan seluruh struktur hierarki komponen terhubung (*density-connected components*) pada graf *mutual reachability* dengan tingkat kestabilan klaster maksimum.
- **Inductive Bias:** Mengasumsikan klaster adalah **wilayah berdensitas tinggi** yang dipisahkan oleh celah berdensitas rendah (*density-based bias*), tidak mengasumsikan bentuk konveks/bulat, dan menganggap titik di luar wilayah padat sebagai *noise*.

## Bab 6.2.3 Pelatihan Parameter Awal & Perbandingan Awal vs Baseline (Validation Set)

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from data_loader import load_splits
from modeling import train_kmeans, train_hdbscan, predict_hdbscan_nearest
from evaluation import evaluate_clustering, compute_spatial_coherence

# Muat Train & Validation Set (Test Set tetap tidak disentuh)
train_df, val_df, _ = load_splits('../data/split')
scaled_cols = [c for c in train_df.columns if c.endswith('_scaled')]
X_train = train_df[scaled_cols].values
X_val = val_df[scaled_cols].values

# Pelatihan HDBSCAN parameter awal
hyperparams_hdb = {'min_cluster_size': 15, 'min_samples': 5, 'cluster_selection_method': 'eom'}
print('Hyperparameter HDBSCAN:', hyperparams_hdb)

hdb_model, hdb_train_lbl = train_hdbscan(X_train, **hyperparams_hdb)
hdb_val_lbl = predict_hdbscan_nearest(hdb_model, X_val)

hdb_train_metrics = evaluate_clustering(X_train, hdb_train_lbl)
hdb_val_metrics = evaluate_clustering(X_val, hdb_val_lbl)
val_tmp = val_df.copy()
val_tmp['cluster'] = hdb_val_lbl
hdb_val_metrics['spatial_coherence'] = compute_spatial_coherence(val_tmp, label_col='cluster')

print('HDBSCAN Train Metrics:', hdb_train_metrics)
print('HDBSCAN Val Metrics  :', hdb_val_metrics)

In [ ]:
# Perbandingan Awal Baseline (K-Means k=4) vs Advanced (HDBSCAN) pada Validation Set
km4, _ = train_kmeans(X_train, n_clusters=4, random_state=42)
km4_val_lbl = km4.predict(X_val)
km4_val_metrics = evaluate_clustering(X_val, km4_val_lbl)
val_tmp['cluster'] = km4_val_lbl
km4_val_metrics['spatial_coherence'] = compute_spatial_coherence(val_tmp, label_col='cluster')

comp_val = pd.DataFrame([
    {'model': 'Baseline: K-Means (k=4)', **km4_val_metrics},
    {'model': 'Advanced: HDBSCAN (15, 5)', **hdb_val_metrics},
])
comp_val